# 1. EDA

El objetivo no es describir columnas: es responder si la **cascada de recomendación** de `overview.md`
es viable con los datos que hay.

| KPI | Qué exige de los datos |
|---|---|
| **Lead Quality Score** | Que el filtro de categoría + modalidad sea correcto y que la similitud tenga insumos reales |
| **Lead Conversion Rate** | Que cada consulta pueda llenar un Top-10 con candidatos genuinamente distintos |

Cada sección mide un nivel de la cascada y cierra con su implicación para el KPI.

In [1]:
import pandas as pd

from src.analysis import eda

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 90)

spots, atributos = eda.cargar_datos()
spots.shape, atributos.shape

((2500, 23), (2500, 11))

---
## 1.1 Contrato de datos

In [2]:
eda.perfil_columnas(spots)

,columna,tipo,pct_nulos,n_unicos,ejemplo
0,venta_total,Float64,86.00,291,65000000.0
1,venta_m2,Float64,86.00,306,21666.67
2,colonia,str,48.72,422,Valle Hermoso
3,estado,str,36.92,26,México
4,corredor,str,31.00,77,Vallejo
5,descripcion,str,16.32,2081,"Bodega en renta sobre Gustavo Baz en Industrial Barrientos, Tlalnepantla. cuenta con: ..."
6,mantenimiento,Float64,9.40,270,0.0
7,renta_m2,Float64,9.24,716,1531035.0
8,renta_total,Float64,9.20,1689,14206473765.0
9,user_id,str,3.72,300,"9,906"


**Hallazgo.** En el CSV crudo todo llega como texto: precios con separador de miles (`"1,531,035"`),
coordenadas con hemisferio (`"99.20000000° W"`), fechas en inglés (`"January 5, 2024"`). Los `float`
que se ven de aquí en adelante existen porque `cargar_datos()` aplica el contrato de tipos; nada se
imputa ni se descarta aquí.

In [3]:
eda.nulos_estructurales(spots)

,columna,pct_nulos_bruto,n_aplica,pct_nulos_donde_aplica,pct_llenos_donde_no_aplica
0,renta_m2,9.24,2270,0.04,0.0
1,renta_total,9.20,2270,0.00,0.0
2,venta_m2,86.00,350,0.00,0.0
3,venta_total,86.00,350,0.00,0.0


**Hallazgo — el 86% de nulos en venta no es un problema de calidad.**
Donde la columna aplica está llena al 100%; donde no aplica, vacía al 100%. El vacío lo determina
`modalidad`, no un defecto del dato.

> **Implicación:** imputar precio de venta en un inmueble de renta inventaría un precio. Estas
> columnas se resuelven por partición, no por imputación.

In [4]:
eda.recuperabilidad_jerarquica(spots, hijo="municipio", padre="estado")

,padre,hijo,n_faltantes_padre,n_recuperables_via_hijo,pct_recuperable,hijos_ambiguos
0,estado,municipio,923,758,82.12,6


**Hallazgo.** `estado` falta en 923 filas, pero `municipio` nunca falta y determina el estado de
forma unívoca en 128 de 134 casos: **82% de los vacíos es recuperable** sin imputar nada. Los 6
"ambiguos" (`Tijuana`, `Mexicali`, `Cuernavaca`, `Saltillo`, `Juárez`, `Miguel Hidalgo`) son en su
mayoría **errores en `estado`**, no homónimos reales.

> **Decisión (2026-09-13).** El modelo usa solo `lat`/`lon` (0% nulos, más precisas que cualquier
> división administrativa). `colonia` (48.7% nulos) se descarta. `corredor` (31.0% nulos) tiene sentido
> de negocio, pero inferirlo desde `lat`/`lon` acierta **87%** (leave-one-out) y sus nulos se
> concentran en Southeast (84% vacío): por debajo del 90% exigido, se descarta.
> `estado` se completa y corrige **en limpieza** con el catálogo INEGI (municipio → estado, CSV
> estático, ~100%), no en inferencia: el modelo recibe datos 100% limpios. `municipio`/`estado` solo
> viajan como texto de salida de la API.

---
## 1.2 Taxonomía — qué columna define "tipo de inmueble"

In [5]:
eda.taxonomia_declarada(spots)

,columna,codigo,significado,n,pct,documentado
0,sector_id,9,Industrial,1036,41.44,True
1,sector_id,11,Office,1236,49.44,True
2,sector_id,12,Retail,0,0.00,True
3,sector_id,13,SIN DOCUMENTAR,212,8.48,False
4,sector_id,15,Land,16,0.64,True
5,tipo_id,1,Single,1064,42.56,True
6,tipo_id,2,Complex,1091,43.64,True
7,tipo_id,3,Subspace,345,13.80,True


**Hallazgo — la clave del Nivel 1 es `sector_id`, y tiene un código sin documentar.**

- `sector_id` = categoría de inmueble: 9 Industrial, 11 Office, 12 Retail, 15 Land.
- `tipo_id` = estructura del espacio: 1 Single, 2 Complex, 3 Subspace. **No es taxonomía de negocio.**
- El diccionario declara `12 → Retail`, pero los datos traen **código 13 (212 filas, 8.5%)** y ningún 12.

columnas a utilizar: Spot Sector ID, Spot Type ID

In [6]:
eda.estructura_espacio(spots)

,sector_id,sector,Single,Complex,Subspace,n,pct_complex
0,9,Industrial,345,350,341,1036,33.78
1,11,Office,538,695,3,1236,56.23
2,13,SIN DOCUMENTAR,165,46,1,212,21.70
3,15,Land,16,0,0,16,0.00


**Hallazgo** — la categoria `Complex`, dentro Spot Type ID (tipo_id), se refiere a un inmueble contenedor (centro comercial completo, el edificio entero, etc), no a una categoria de inmueble individual. Ambas categorias **NO** son comparables por obvias razones. 

**Accionable:** **EXCLUIR** esos inmuebles contenedores. crear una columna nueva (por ejemplo `es_candidato = False`). se excluye del modelado más no se elimina las entradas del CSV.

### Accionables — 1.2 Taxonomía

**Columnas:** `Spot Sector ID` → `sector_id`, `Spot Type ID` → `tipo_id`.

**1. `sector_id` es la clave taxonómica; `tipo_id` no lo es.** `sector_id` define la categoría de
negocio (Industrial / Office / Retail / Land) sobre él se
construye la partición del Nivel 1 (ver 1.3). `tipo_id` **no es dimensión de partición, sino filtro de elegibilidad** (accionable 3 de exclusión solamente)

**2. Mapear `sector_id 13 → 12 (Retail)`.**

**3. Excluir los `Complex` como candidatos (43.6%).**

Los tres se ejecutan en la **etapa 2 (limpieza)**

## 1.3 Nivel 1 — filtro por categoría y modalidad

**hallazgo** 
- Se tiene que hacer un corte de sector_id (Spot Sector ID) × modalidad (Spot Modality), es decir categoría × intención.
- De ese corte salen 8 grupos:: 4 sectores con datos (Industrial, Office, SIN DOCUMENTAR, Land) × 2 intenciones (Rent, Sale)

 **problema** : 
 - Land-Rent tiene 2 inmuebles → 1 candidato útil (el otro es la consulta misma). No llega a 10.
- Los otros 7 grupos tienen inventario de sobra (el más chico, Land-Sale, tiene 16).

**accionable**
- eliminar que se tiene que retornar un top 10. es decir, se devuelve lo que haya (o no) y en caso hayan mas de 10, entonces ahi recien se selecciona los mejores 10. 

In [7]:
eda.etiquetar_sector(eda.matriz_particiones(spots, min_n=eda.MIN_CANDIDATOS))

,sector_id,sector,intencion,n_inventario,n_candidatos_max,viable,pct_inventario
0,15,Land,Rent,2,1,False,0.08
1,15,Land,Sale,16,15,True,0.64
2,13,SIN DOCUMENTAR,Sale,42,41,True,1.68
3,9,Industrial,Sale,120,119,True,4.80
4,11,Office,Sale,172,171,True,6.88
5,13,SIN DOCUMENTAR,Rent,180,179,True,7.20
6,9,Industrial,Rent,933,932,True,37.32
7,11,Office,Rent,1155,1154,True,46.20


In [8]:
eda.cobertura_filtro_nivel1(spots, min_n=eda.MIN_CANDIDATOS)

,min_candidatos,n_particiones,n_particiones_inviables,pct_leads_servibles
0,10,8,1,99.92


---
## 1.4 Nivel 2 — BallTree geográfico


In [9]:
eda.validar_coordenadas(spots)

,n_registros,pct_nulos,pct_fuera_de_mexico,n_coordenadas_unicas,pct_coordenada_compartida,decimales_mediana,resolucion_aprox_km
0,2500,0.0,0.0,871,65.16,2,1.11


**Hallazgo — las coordenadas están redondeadasa 2 decimales** 

- significa que las coordenadas pueden tratar erroneamente como "cercanos" a dos o mas inmueble cuando en realidad estan lejos. todo porque las coordenas estan redondeadas


In [10]:
densidad = eda.densidad_vecinos(spots, radios_km=eda.RADIOS_KM)
eda.cobertura_radio(densidad, min_n=eda.MIN_CANDIDATOS)

,radio_km,pct_alcanza_min_n,pct_sin_ningun_vecino,mediana_vecinos
0,5.0,71.87,6.76,35.5
1,10.0,83.63,4.35,100.5
2,15.0,87.10,3.55,138.0


In [11]:
eda.etiquetar_sector(
    eda.cobertura_radio_por_particion(densidad, min_n=eda.MIN_CANDIDATOS, radio_km=15.0)
)

,sector_id,sector,intencion,n_consultas,mediana_vecinos,pct_alcanza_min_n
0,13,SIN DOCUMENTAR,Sale,42,2.0,0.00
1,15,Land,Rent,2,0.0,0.00
2,15,Land,Sale,16,0.0,0.00
3,9,Industrial,Sale,120,8.0,45.83
4,13,SIN DOCUMENTAR,Rent,180,21.0,69.44
5,9,Industrial,Rent,933,67.0,88.00
6,11,Office,Sale,172,133.0,90.12
7,11,Office,Rent,1155,1005.0,97.49


# Mostrar un top 10 de inmuebles similares es viable?

**Hallazgo**: El 87.1% es: de todos los inmuebles, cuántos tienen al menos 10 vecinos de su misma partición dentro de 15 km (radio tope). Es decir, para el 87.1% de los clics, el Nivel 2 consigue suficientes candidatos para llenar un Top-10.
### El 87.1% es un promedio que miente

Ese número global está dominado por Office Rent, que solo, es el 46% del inventario. Cuando lo abres por partición:

(mirar la tabla de arriba)

Tres comportamientos distintos:

- Office Rent ya está resuelto a 5 km. Estirar el radio no le aporta nada (+1.4).
- Las del medio mejoran, pero pagando: a 15 km ya estás comparando Polanco con Ecatepec.
- Tres particiones no se mueven del 0%. Ni un milímetro (los primeros de la tabla de arriba)

---

## La causa real no es el radio: es la densidad

Mira inmuebles por municipio:

| partición | inmuebles | municipios | por municipio |
|---|---:|---:|---:|
| Office Rent | 1155 | 40 | 28.9 |
| Industrial Sale | 120 | 54 | 2.2 |
| sector 13 Sale | 42 | 25 | 1.7 |
| Land Sale | 16 | 16 | 1.0 |

Office Rent tiene 293 oficinas solo en Miguel Hidalgo. Te sobran vecinos sin salir de la colonia.

sector 13 Sale tiene 42 locales repartidos en 25 municipios — Guadalajara 3, Querétaro 3, Solidaridad 3, y el resto de a uno. No hay a quién comparar, no importa cuánto abras el círculo.

---

# Industrial Sale: el caso interesante

Mediana de 8 vecinos a 15 km. Casi llega, pero no. Y está partido en dos:

- **14 inmuebles** → 0 vecinos (municipio aislado)
- **25% inferior** → ≤3 vecinos
- **25% superior** → ≥30 vecinos (corredor Querétaro / Naucalpan / Tlalnepantla)

No hay un radio que sirva para los dos grupos. Si lo pones para rescatar a los 14 aislados, a los del corredor les metes basura de otra ciudad.

---

# Accionable — N es un máximo, no un mínimo

**Decidido 2026-09-15.** El sistema no llena un Top-10; devuelve **hasta 10**.

Regla: buscar en la partición dentro del radio (5 → 10 → 15 km), ordenar por `sim`, devolver `min(10, encontrados)`. La respuesta informa cuántos encontró. Encontrar 1 es un resultado válido.

Sin N por partición, sin fórmula, sin α. La cobertura por partición (tabla de arriba) se reporta como métrica en Etapa 6, no como regla del sistema.


---
## 1.5 Nivel 3A — insumos de sim(A,B)

La similitud es `w_tab · cos(tab) + w_nlp · cos(nlp)`. Si a un candidato le falta una mitad, compite
en desventaja por un defecto del dato, no por ser peor opción.

In [12]:
eda.etiquetar_sector(eda.cobertura_texto(spots))

,sector_id,sector,intencion,n,pct_sin_descripcion,pct_texto_pobre,mediana_caracteres
0,9,Industrial,Rent,933,36.12,36.12,256.0
1,11,Office,Sale,172,6.40,6.40,256.0
2,11,Office,Rent,1155,5.80,6.23,272.0
3,9,Industrial,Sale,120,2.50,2.50,313.5
4,13,SIN DOCUMENTAR,Rent,180,0.00,0.00,369.5
5,13,SIN DOCUMENTAR,Sale,42,0.00,0.00,414.0
6,15,Land,Rent,2,0.00,0.00,391.5
7,15,Land,Sale,16,0.00,0.00,396.5


**Hallazgo — el hueco de viene del texto. La descripcion faltante proviene de Industrial Rent: 36.1% sin descripción.** El resto de
particiones va de 0% a 6.4%.

> **Implicación:** un texto vacío no da un vector cero, da uno fijo: dos naves sin descripción se
> parecen entre sí por no tener texto, y una sin texto pierde contra una buena. `w_nlp` no puede ser global.

> **Accionable:** pesos por par, renormalizados sobre las señales presentes en **ambos** lados —
> `sim = Σ(wᵢ·cosᵢ) / Σ(wᵢ)`. Con texto: `(0.6·tab + 0.4·nlp)/1.0`. Sin texto: `0.6·tab/0.6 = cos_tab`.
> Renormalizar, no truncar: así el score sigue en [0,1] y el umbral del Top-N significa lo mismo en
> toda la partición. Se devuelve `señales_usadas` y se evalúa por separado el grupo sin texto.

In [13]:
eda.duplicados_texto(spots)

,columna,n_con_texto,n_textos_unicos,pct_en_texto_repetido,max_repeticiones
0,titulo,2500,2333,10.32,11
1,descripcion,2092,2081,0.86,5


In [14]:
eda.dimensiones_usables_por_sector(spots, atributos, objetivo=5)

,sector_id,sector,n_dimensiones,columnas,alcanza_objetivo
0,9,Industrial,6,"area_m2, mantenimiento, puertos_carga, estado_edificio, luz_natural, material_piso",True
1,11,Office,5,"area_m2, mantenimiento, estado_edificio, tipo_seguridad, elevadores",True
2,13,SIN DOCUMENTAR,4,"area_m2, estacionamientos, mantenimiento, estado_edificio",False
3,15,Land,1,area_m2,False


**Hallazgo.** Los inmuebles son de distinta naturaleza según el sector (oficina, nave, terreno), y por eso no comparten las mismas columnas tabulares: una oficina no tiene puertos de carga, un terreno no tiene elevadores. Esos vacíos son estructurales, no datos faltantes. Con los datos limpios, los vectores numéricos que salen de la data tabular no tendrán las mismas dimensiones en todos los sectores:

| Sector | Dimensiones ≥50% cobertura |
|---|---|
| Industrial | **6** — área, mantenimiento, puertos de carga, estado del edificio, luz natural, material de piso |
| Office | **5** — área, mantenimiento, estado del edificio, tipo de seguridad, elevadores |
| Sector 13 | 4 — área, estacionamientos, mantenimiento, estado del edificio |
| Land | **1** — solo área |

> **Accionable.** Etapa 2: no imputar esos vacíos; rellenarlos inventa un atributo que el inmueble no tiene y contamina la información. Etapa 4: un vector tabular por sector, escalado dentro del sector, para que las dimensiones reflejen las características cuantitativas reales. Land (solo área, 18 inmuebles) no tiene vector tabular; su `sim` usa solo NLP.


---
## 1.6 Nivel 3B — f(r) sobre precio por m²

In [15]:
eda.consistencia_precio(spots)

,modalidad,n_evaluable,pct_identidad_ok,error_relativo_mediano
0,renta,2269,99.69,0.0
1,venta,350,100.00,0.0


**Hallazgo.** La identidad `total = precio_m² × área` se cumple en 99.7% (renta) y 100% (venta). Las
columnas de precio son internamente coherentes y `precio_m²` puede alimentar `f(r)` directamente.

In [16]:
eda.rango_precio_m2(spots)

,intencion,count,mean,std,min,1%,25%,50%,75%,99%,max
0,Rent,2269.0,2028.7,36227.6,0.0,5.7,130.0,250.0,395.0,9717.9,1531035.0
1,Sale,350.0,45632.8,105567.8,1.0,20.0,12543.2,27204.2,56885.9,181136.1,1271929.4


In [17]:
eda.etiquetar_sector(eda.outliers_precio(spots, k=1.5))

,sector_id,sector,intencion,n,n_outliers,pct_outliers,limite_inferior,limite_superior,max_observado
0,9,Industrial,Sale,120,15,12.50,2068.3,67558.9,109693.0
1,11,Office,Sale,172,18,10.47,13373.4,202784.0,1271929.4
2,11,Office,Rent,1154,92,7.97,157.5,878.0,285000.0
3,15,Land,Sale,16,1,6.25,207.6,99460.6,80132.4
4,9,Industrial,Rent,933,56,6.00,54.7,274.7,450000.0
5,13,SIN DOCUMENTAR,Rent,180,10,5.56,54.0,1779.8,1531035.0
6,13,SIN DOCUMENTAR,Sale,42,1,2.38,4273.9,213618.3,188733.7
7,15,Land,Rent,2,0,0.00,0.0,5541.3,85.0


**Hallazgo.** La escala es correcta en el grueso (renta: mediana 250 MXN/m²; venta: mediana 27.204
MXN/m²), pero la cola es imposible: **1.531.035 MXN/m² de renta** en el máximo. Entre 2.4% y 12.5%
de cada partición cae fuera del rango intercuartil en escala logarítmica.

> **Implicación:** `f(r)` es un cociente. Si el outlier cae en el denominador (el inmueble
> clickeado), todos sus vecinos se ven regalados o ridículamente caros, y el orden de esa consulta
> se distorsiona entero. Por eso hay que limpiar **antes** de calibrar la cota `[0.7, 1.1]`: sobre
> una distribución contaminada, la cota quedaría ajustada al ruido.

> **Accionable:** el recorte se decide en limpieza. Alternativas: **IQR sobre log** por partición,
> **tope duro de mercado** por sector/modalidad, o **winsorización** (p1–p99). Aquí solo se cuantifica.

In [18]:
eda.distribucion_ratio_precio(spots, radio_km=10.0)

,n_pares,radio_km,p1,p10,p25,p50,p75,p90,p99,pct_r_fuera_de_0.5_2.0
0,266888,10.0,0.007,0.471,0.722,1.0,1.385,2.128,150.32,22.27


**Hallazgo 3 — el rango de `f(r)` no corresponde a la distribución observada de `r`.** Se calculó la
razón de precios `r = P_m²(B)/P_m²(A)` sobre 266.888 pares del catálogo comparables entre sí (misma
partición, ≤10 km). La distribución está centrada (mediana = 1.0) pero es ancha: el **80% central de
los pares vive en `r ∈ [0.47, 2.13]`** (p10–p90), y el p99 llega a 150.

> **Implicación:** `f(r)` está truncada en `[0.7, 1.1]`, un rango definido a priori. Como la
> dispersión real de `r` es mucho mayor que ese supuesto, el truncamiento alcanza a una fracción
> grande de los pares: un candidato en p90 (`r = 2.13`, el doble de caro pero comparable) y uno en
> p99 (`r = 150`, precio atípico) reciben idéntico valor (0.70). El factor de precio se vuelve
> constante y deja de aportar varianza al ordenamiento.

> **Accionable:** (1) tratar los valores atípicos de precio; (2) reestimar los percentiles de `r` por
> partición, ya que la dispersión de Industrial no es la de Office; (3) definir el rango de `f(r)` a
> partir de esos percentiles, de modo que p10–p90 quede sin truncar y solo se trunquen las colas. Los
> atípicos explican poco: el rango deberá ampliarse de todos modos.

---
## 1.7 Nivel 3C — ¿hace falta el k-means de diversidad?

In [19]:
eda.etiquetar_sector(eda.tasa_clones(spots, tolerancia_area=0.05, radio_km=1.0))

,sector_id,sector,intencion,n,n_con_al_menos_un_clon,pct_con_clon,max_clones_de_un_inmueble
0,9,Industrial,Rent,933,292,31.30,14
1,11,Office,Rent,1154,236,20.45,10
2,11,Office,Sale,172,13,7.56,8
3,9,Industrial,Sale,120,8,6.67,1
4,13,SIN DOCUMENTAR,Rent,180,2,1.11,1
5,13,SIN DOCUMENTAR,Sale,42,0,0.00,0
6,15,Land,Rent,2,0,0.00,0
7,15,Land,Sale,16,0,0.00,0


#### Contexto

`models.md` propone un **K-Means para diversificar el Top-N**.

Un modelo solo se justifica si el problema que arregla existe, así que había que medirlo antes de construirlo.

---

####  Qué se midió

**"Clones"**: inmuebles del mismo usuario, a ≤ 1 km, con área equivalente (±5%).

Es decir, publicaciones prácticamente indistinguibles entre sí.

---

####  El problema

**21.0% del inventario tiene al menos un clon.**

La concentración es especialmente alta en algunas particiones:

| Partición | % con clon | Máx. clones de un inmueble |
|:--|--:|--:|
| **Industrial Rent** | **31.3%** | **14** |
| **Office Rent** | **20.5%** | **10** |
| Resto | 0–7.6% | ≤8 |

####  Ejemplo

Un parque industrial publica sus **14 bodegas iguales por separado**.

El usuario hace clic en una de ellas. Como `sim` mide parecido, las otras 13 son las más parecidas que existen:

> **coseno ≈ 1**

Por lo tanto, el **Top-10 se llena con la misma nave diez veces**.

Técnicamente, el ranking es perfecto: las propiedades realmente son muy similares.

Comercialmente, sin embargo, el usuario **solo vio una opción** y se fue.

---

#### Conclusión

El **K-Means queda justificado por evidencia, no por intuición**.

Y actúa en el lugar correcto:

1. Primero se ordena por `sim`.
2. Después, al construir el corte Top-N, se penaliza repetir cluster.
3. `sim` no se modifica.

Esto último es importante: si el clustering sale mal, el **Top-N simplemente degrada hacia el orden original**; el ranking base no se rompe.

---
## 1.8 Veredicto

In [20]:
eda.veredicto_cascada(spots, atributos, min_n=eda.MIN_CANDIDATOS)

,nivel,metrica,valor,bloqueante
0,1 - tipo + modalidad,% de leads en particion servible,99.92,True
1,2 - BallTree geografico,% que alcanza 10 candidatos a 15 km,87.10,True
2,3A - sim NLP,% de inventario sin descripcion,16.32,True
3,3A - sim tabular,dimensiones tabulares en el sector peor cubierto,1.00,True
4,3B - f(r) precio,% donde total = precio_m2 x area (peor modalidad),99.69,False
5,3C - k-means diversidad,% de inventario con al menos un clon,21.04,False


### Conclusión

Los datos **soportan la cascada**, con seis condiciones que resolver antes de modelar:

1. **Taxonomía.** La clave del Nivel 1 es `sector_id`, no `tipo_id`. El código 13 (8.5%) no está en el diccionario.
2. **Estructura.** 43.6% son `Complex` (contenedores), que el PDF excluye del ejercicio.
3. **Nivel 1.** `(Land, Rent)` tiene 2 inmuebles: necesita fallback explícito.
4. **Nivel 2.** 87.1% de cobertura al radio tope. El Top-10 universal no existe; el N debe ser variable.
5. **Nivel 3A.** Industrial Rent tiene 36.1% sin texto, y el vector tabular es distinto por sector (6 / 5 / 4 / 1).
6. **Nivel 3B.** La identidad de precios es sólida, pero los outliers rompen `r` y saturan `f(r)`.

Lo que **no** es problema: los nulos de venta (estructurales), el estado faltante (82% recuperable
desde municipio) y la coherencia de precios (99.7%).

**Pasa a la etapa 2 (limpieza):** ver `HANDOFF.md`.